# 03 — Prompt Engineering

## 📚 Learning Objectives

By completing this notebook, you will:
- Know the main **prompt-engineering strategies**: zero-shot, few-shot, chain-of-thought, role prompting
- See hands-on how **conditioning text steers a language model**: the same trained model produces different continuations for different seeds and temperatures
- Have working **reference templates** for the OpenAI API and Hugging Face `pipeline` (shown as reference; running them needs an API key / model download)

## 🔗 Prerequisites

- ✅ Example 01 (character-level language model — reused for the hands-on part)

---

## Introduction

A language model only ever does one thing: continue the text it is given. **Prompt engineering** is the craft of writing that text so the continuation is what you want. With instruction-tuned LLMs the prompt can carry instructions, examples, and reasoning cues; with our small char-LM the prompt is just a seed — but the *mechanism* (conditioning changes the output distribution) is the same one, and we can demonstrate it live.


## The Four Core Strategies

**Zero-shot** — instruction only:
```text
Classify this review's sentiment as positive or negative.
Review: "The battery dies in an hour."
Sentiment:
```

**Few-shot** — show worked examples first; the model imitates the pattern:
```text
Review: "Amazing sound quality!"        → positive
Review: "Broke after two days."         → negative
Review: "The battery dies in an hour."  →
```

**Chain-of-thought** — ask for reasoning steps before the answer (helps on math/logic):
```text
Q: A shop had 23 apples, sold 9, bought 15. How many now?
A: Let's think step by step.
```

**Role prompting** — set persona and audience to control tone and depth:
```text
You are a patient math tutor for 12-year-olds. Explain fractions with one everyday example.
```

Practical rules that hold across all of them: be specific, state the output format, give the model examples of exactly what you want, and iterate — prompt design is empirical.


In [1]:
# WHAT/WHY: demonstrate the MECHANISM behind prompting — conditioning text
# steers generation. We train one char-level LM, then vary only the seed text
# and the temperature, and watch the continuations change.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)
chars = sorted(set(text)); c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}; VOCAB = len(chars); SEQ_LEN = 20
enc = [c2i[c] for c in text]
X = torch.tensor([enc[i:i+SEQ_LEN] for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)
y = torch.tensor([enc[i+SEQ_LEN]   for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)

# ── Train the LM once (same model as example 01) ──────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

model = CharLM(); opt = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()
for step in range(300):
    model.train()
    perm = torch.randperm(len(X))[:256]
    loss = loss_fn(model(X[perm]), y[perm])
    opt.zero_grad(); loss.backward(); opt.step()
print(f"LM trained — final loss {loss.item():.3f}")

def generate(seed, steps=70, temperature=0.8):
    # condition on the seed, then sample the continuation char by char
    model.eval()
    out = list(seed); ctx = [c2i.get(c, 0) for c in seed[-SEQ_LEN:]]
    for _ in range(steps):
        with torch.no_grad():
            logits = model(torch.tensor([ctx[-SEQ_LEN:]]))[0] / temperature
        nxt = int(np.random.choice(VOCAB, p=torch.softmax(logits, 0).numpy()))
        out.append(i2c[nxt]); ctx.append(nxt)
    return ''.join(out)

# ── Steering experiment 1: SAME model, different seeds ────────────────────
# Seeds are exact 20-char slices of the corpus (the context length the model
# was trained with), taken from three different passages.
np.random.seed(7)
print("\n— Different seeds steer the continuation (temperature 0.8) —")
for seed in [text[0:20], text[120:140], text[230:250]]:
    print(f"  seed {seed!r}\n    → {generate(seed)!r}")

# ── Steering experiment 2: SAME seed, different temperatures ──────────────
np.random.seed(7)
print("\n— Same seed, different temperatures (decoding is part of the prompt kit) —")
for temp in [0.3, 1.5]:
    print(f"  temperature {temp}:\n    → {generate(text[0:20], temperature=temp)!r}")

print("\nWhat this shows: generation is conditioned on the text you provide.")
print("Prompt engineering on an LLM exploits exactly this — with instructions and")
print("examples in the conditioning text instead of a bare seed. Note the limits:")
print("a char-LM cannot follow instructions; that ability comes from instruction")
print("tuning (RLHF) on large models.")


LM trained — final loss 0.003

— Different seeds steer the continuation (temperature 0.8) —
  seed 'to be or not to be t'
    → 'to be or not to be that is the question whether tis nobler in the mind to suffer the sling'
  seed 'tune or to take arms'
    → 'tune or to take arms against a sea of troubles and by opposing end them to die to sleep no'
  seed ' to say we end the h'
    → ' to say we end the heartache and the thousand natural shocks that flesh is heir to tis a c'

— Same seed, different temperatures (decoding is part of the prompt kit) —
  temperature 0.3:
    → 'to be or not to be that is the question whether tis nobler in the mind to suffer the sling'
  temperature 1.5:
    → 'to be or not to be that is the question whether tis nobler in the mind to staiy sskamrofrh'

What this shows: generation is conditioned on the text you provide.
Prompt engineering on an LLM exploits exactly this — with instructions and
examples in the conditioning text instead of a bare seed. Note th

## The Same Strategies on Real LLMs (reference)

**Reference only — not executed here** (needs an API key or a model download):

OpenAI API — the prompt strategies go into the `messages`:
```python
from openai import OpenAI
client = OpenAI()   # needs OPENAI_API_KEY

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a patient math tutor."},   # role prompt
        {"role": "user",   "content": "Q: 23 apples, sold 9, bought 15. "
                                       "How many now? Think step by step."}  # chain-of-thought
    ],
    temperature=0.3,   # same knob you used above
)
print(resp.choices[0].message.content)
```

Hugging Face `pipeline` — few-shot by stacking examples into the prompt:
```python
from transformers import pipeline
generator = pipeline("text-generation", model="distilgpt2")   # downloads the model

prompt = ("Review: 'Amazing sound!' → positive\n"
          "Review: 'Broke in two days.' → negative\n"
          "Review: 'Battery dies fast.' →")
print(generator(prompt, max_new_tokens=5, temperature=0.3)[0]["generated_text"])
```

Try these on your own machine/account — then iterate on the prompt and watch the output change, exactly as in the steering experiment above.


## 📚 References & Further Reading

**Papers:**
- Brown et al. (2020) — [GPT-3: Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) *(few-shot prompting)*
- Wei et al. (2022) — [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)

**Guides:**
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [promptingguide.ai](https://www.promptingguide.ai/) *(open catalogue of techniques)*


## 📝 Summary

In **03 — Prompt Engineering** you learned the four core strategies (zero-shot, few-shot, chain-of-thought, role prompting) with concrete templates, and demonstrated the underlying mechanism on a model you trained: changing only the conditioning text (seed) or the temperature changed the generation — see the printed experiments. The API reference blocks show how the same strategies are written for GPT-class models; running them requires an API key or model download, which this classroom notebook deliberately avoids.
